# Sensitivity of filter feeders to TEP excretion (`p_pueR2`, `p_pueR3`)

Suspension feeders (`Y3`, and juveniles `Yy3`) take up two kinds of TEP:

* **R2**, free TEP, taken up together with diatoms (`P1`, `P5`) in proportion to the
  diatoms eaten. A fraction `p_pueR2` of it is excreted again (as faeces, to `R6`).
* **R3**, the carbon in *Phaeocystis* colonies outside the cells (the colony matrix).
  `Y3` eats *Phaeocystis* (`P6`; `p_puPI(1,6) = 1`) and takes up the colony's R3 with it.
  A fraction `p_pueR3` of it is excreted.

Both fractions are 0.90 in the OysterGrounds case. They also set the overall assimilation
efficiency of the food (`sunmc` in `FilterFeeder.F90`), which decides whether filtering is
worth its cost (`active`). A lower excretion fraction therefore gives more carbon from TEP,
and also more filtering of all food, mainly during the *Phaeocystis* bloom.

This notebook varies each fraction from 0.90 to 0.10 while the other stays at 0.90:

* `p_pueR2` = 0.90, 0.40, 0.20, 0.10 (with `p_pueR3` = 0.90)
* `p_pueR3` = 0.90, 0.40, 0.20, 0.10 (with `p_pueR2` = 0.90)

Both are namelist parameters in `FilterFeeder.nml` (`/FilterFeeder_parameters/`), with one
value per filter-feeder group (`Y3`, `Yy3`), so **no recompilation is needed**.
Do not use exactly 0 for `p_pueR2`: the model takes that as "no TEP uptake".

Outputs, over one year:

* **(a)** annual mean `Y3c` against the excretion fraction, one line for `p_pueR2` and
  one for `p_pueR3`, labelled with the % change from 0.90/0.90
* **(b)** daily `Y3c` through the year: `p_pueR2` runs solid, `p_pueR3` runs dashed

One year takes about 6 minutes per run; the runs are started in parallel.

## NaaVRE workflow

```
[TEP scenarios] --> [run GOTM TEP] --> [plot TEP]
   param_scenarios: List   run_dirs: List   results_folder: String
```

1. **TEP scenarios**: the list of scenarios, items `"pueR2/pueR3"`.
2. **run GOTM TEP**: one model run per scenario, in parallel; outputs `run_dirs`, the run
   folders under `/tmp/data/` (the only folder shared between components).
3. **plot TEP**: reads `OG.nc` from each folder, makes the table and figure and saves them,
   with each run's `log.out` and `run_info.json`, to Cloud Storage, because NaaVRE clears
   `/tmp/data/` when the workflow finishes. It outputs that folder as `results_folder`.


In [ ]:
# TEP scenarios
# ---
# NaaVRE:
#  cell:
#   inputs: []
#   outputs:
#    - param_scenarios: List
# ...
# One item per model run: "pueR2/pueR3" (excreted fraction of TEP uptake, R2 and R3).
# The case default is 0.90/0.90; it is used as the reference for the % change.
param_scenarios = [
    "0.90/0.90",                              # reference (case default)
    "0.40/0.90", "0.20/0.90", "0.10/0.90",    # p_pueR2 varied, p_pueR3 = 0.90
    "0.90/0.40", "0.90/0.20", "0.90/0.10",    # p_pueR3 varied, p_pueR2 = 0.90
]
print("param_scenarios:", param_scenarios)


In [ ]:
# run GOTM TEP
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - param_scenarios: List
#   outputs:
#    - run_dirs: List
#   params:
#    - param_year:
#       type: Integer
#       default_value: 1972
#    - param_var:
#       type: String
#       default_value: "Y3c"
#    - param_start:
#       type: String
#       default_value: "1972-01-01 00:00:00"
#    - param_stop:
#       type: String
#       default_value: "1973-01-01 00:00:00"
# ...


# Settings (do not containerize this cell)
# param_ values are the defaults of the workflow components; they can be changed when
# the workflow is submitted.
param_start = "1972-01-01 00:00:00"
param_stop = "1973-01-01 00:00:00"
param_results_root = "/tmp/data/TEPexcretion"   # shared between workflow components
param_year = 1972            # year to average over
param_var = "Y3c"            # BFM variable in OG.nc, e.g. "jnetY3c" for net production
param_cloud_folder = "/home/jovyan/Cloud Storage/naa-vre-user-data/TEPexcretion"   # kept after the workflow

# TEP scenarios: labels "pueR2_<x>_pueR3_<y>", one per model run.
# param_scenarios (List, from the TEP scenarios cell): items "pueR2/pueR3", e.g. "0.40/0.90",
# or pairs [pueR2, pueR3]; a single number sets pueR2 only (pueR3 stays 0.90).
items = param_scenarios if isinstance(param_scenarios, (list, tuple)) else str(param_scenarios).split(",")
scenarios = []
for item in items:
    if isinstance(item, (list, tuple)):
        values = [float(v) for v in item]
    elif str(item).strip():
        values = [float(v) for v in str(item).split("/")]
    else:
        continue
    pueR2 = values[0]
    pueR3 = values[1] if len(values) > 1 else 0.90
    scenarios.append("pueR2_{:.2f}_pueR3_{:.2f}".format(pueR2, pueR3))
print("scenarios:", scenarios)

# run GOTM TEP
# Output run_dirs (List of str): run folders under param_results_root.
import os
import re
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor


def run_one(label):
    """One GOTM-BFM run for a label "pueR2_<x>_pueR3_<y>"; returns the run folder."""
    parts = label.split("_")
    pueR2, pueR3 = float(parts[1]), float(parts[3])
    run_dir = os.path.join(param_results_root, label)
    if os.path.exists(run_dir):          # gotm-run needs a new or empty directory
        shutil.rmtree(run_dir)
    os.makedirs(param_results_root, exist_ok=True)
    command = [
        "gotm-run",
        "--out", run_dir,
        "--start", param_start,
        "--stop", param_stop,
        # one value per filter-feeder group (Y3, Yy3)
        "--set", "FilterFeeder.nml:p_pueR2={0:g},{0:g}".format(pueR2),
        "--set", "FilterFeeder.nml:p_pueR3={0:g},{0:g}".format(pueR3),
    ]
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    status = "finished" if proc.returncode == 0 else "FAILED (see {}/log.out)".format(run_dir)
    # namelist echo in log.out: the values the model actually used
    used = ""
    log_file = os.path.join(run_dir, "log.out")
    if os.path.exists(log_file):
        with open(log_file, errors="replace") as f:
            used = " | ".join(" ".join(u.split()) for u in
                              re.findall(r"P_PUER[23]\s*=[^A-Z]*", f.read(), flags=re.IGNORECASE))
    print("{:<24} {}   {}".format(label, status, used or "(values not found in log.out)"), flush=True)
    if proc.returncode != 0:
        print(proc.stdout.decode(errors="replace")[-2000:])
    return run_dir


print("Starting {} run(s) from {} to {} ...".format(len(scenarios), param_start, param_stop), flush=True)
with ThreadPoolExecutor(max_workers=max(1, len(scenarios))) as pool:
    run_dirs = list(pool.map(run_one, scenarios))
print("run_dirs:", run_dirs)

scenarios: ['pueR2_0.90_pueR3_0.90', 'pueR2_0.40_pueR3_0.90', 'pueR2_0.20_pueR3_0.90', 'pueR2_0.10_pueR3_0.90']
Starting 4 run(s) from 1972-01-01 00:00:00 to 1973-01-01 00:00:00 ...


In [ ]:
# plot TEP
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - run_dirs: List
#   outputs:
#    - results_folder: String
# ...

# plot TEP
# Input run_dirs (List of str): run folders from the run cell.
# Output results_folder (str): Cloud Storage folder with the figure and the table.
import datetime
import json
import os
import shutil

import matplotlib.pyplot as plt
import netCDF4
import numpy as np

# Flatten, in case run_dirs arrives as a list of lists
folders = []
for item in (run_dirs if isinstance(run_dirs, (list, tuple)) else [run_dirs]):
    folders.extend(item if isinstance(item, (list, tuple)) else [item])

year = int(param_year)
var = str(param_var)

results = []    # (pueR2, pueR3, dates, values)
units, long_name = "", var
for run_dir in folders:
    label = os.path.basename(os.path.normpath(run_dir))
    info_file = os.path.join(run_dir, "run_info.json")
    nc_file = os.path.join(run_dir, "OG.nc")
    if os.path.exists(info_file):
        with open(info_file) as f:
            if json.load(f).get("status") != "finished":
                print("skipped (run failed):", run_dir)
                continue
    if not os.path.exists(nc_file):
        print("skipped (no OG.nc):", run_dir)
        continue
    parts = label.split("_")              # pueR2_<x>_pueR3_<y>
    pueR2, pueR3 = float(parts[1]), float(parts[3])
    with netCDF4.Dataset(nc_file) as nc:
        t = nc.variables["time"]
        # num2date may return cftime dates, which matplotlib cannot plot:
        # convert them to standard Python datetimes
        dates = [datetime.datetime(d.year, d.month, d.day, d.hour, d.minute, d.second)
                 for d in netCDF4.num2date(t[:], t.units)]
        values = np.ma.filled(nc.variables[var][:].reshape(len(dates), -1)[:, 0], np.nan)
        units = getattr(nc.variables[var], "units", "")
        long_name = getattr(nc.variables[var], "long_name", var)
    keep = np.array([d.year == year for d in dates])
    results.append((pueR2, pueR3, [d for d, k in zip(dates, keep) if k], values[keep]))
if not results:
    raise RuntimeError("no finished runs in {}".format(folders))
results.sort(key=lambda r: (-r[0], -r[1]))     # from high to low excretion

# Annual mean and % change from the case default (pueR2 = pueR3 = 0.90, or the first run)
labels = ["R2 {:.2f}, R3 {:.2f}".format(r[0], r[1]) for r in results]
means = np.array([np.nanmean(r[3]) for r in results])
i_ref = next((i for i, r in enumerate(results)
              if abs(r[0] - 0.9) < 1e-6 and abs(r[1] - 0.9) < 1e-6), 0)
ref = means[i_ref]
ref_r2, ref_r3 = results[i_ref][0], results[i_ref][1]
change = 100.0 * (means - ref) / ref

table = ["p_pueR2,p_pueR3,mean_{},change_percent".format(var)]
print("reference:", labels[i_ref])
print("{:>7}  {:>7}  {:>12}  {:>8}".format("p_pueR2", "p_pueR3", "mean " + var, "change"))
for r, m, c in zip(results, means, change):
    print("{:>7.2f}  {:>7.2f}  {:>12.2f}  {:>+7.2f}%".format(r[0], r[1], m, c))
    table.append("{:g},{:g},{:.6g},{:.4f}".format(r[0], r[1], m, c))

# Sweeps: R2 varied (R3 at the reference value), R3 varied (R2 at the reference value);
# runs with both changed are plotted as single points
sweeps = [
    ("p_pueR2 varied (p_pueR3 = {:.2f})".format(ref_r3), 0, 1, "tab:blue", "-", "o"),
    ("p_pueR3 varied (p_pueR2 = {:.2f})".format(ref_r2), 1, 0, "tab:orange", "--", "s"),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
plotted = set()
for name, i_var, i_fix, color, style, marker in sweeps:
    idx = [i for i, r in enumerate(results) if abs(r[i_fix] - results[i_ref][i_fix]) < 1e-6]
    idx.sort(key=lambda i: results[i][i_var])
    if len(idx) < 2:
        continue
    plotted.update(idx)
    xs = [results[i][i_var] for i in idx]
    ax1.plot(xs, means[idx], style, marker=marker, color=color, label=name)
    for i in idx:
        if i != i_ref:
            ax1.annotate("{:+.1f}%".format(change[i]), (results[i][i_var], means[i]),
                         textcoords="offset points", xytext=(0, 8), ha="center",
                         fontsize=8, color=color)
    # (b) daily values: darker = lower excretion fraction
    shades = np.linspace(1.0, 0.45, len(idx))
    for i, shade in zip(idx[::-1], shades):
        if i == i_ref and i_var == 1:
            continue                                # reference drawn once, with the R2 sweep
        r = results[i]
        lab = "reference 0.90/0.90" if i == i_ref else \
            "{} = {:.2f}".format("p_pueR2" if i_var == 0 else "p_pueR3", r[i_var])
        rgb = np.array(plt.matplotlib.colors.to_rgb(color))
        ax2.plot(r[2], r[3], style if i != i_ref else "-", label=lab,
                 color="black" if i == i_ref else rgb * shade, lw=1.2)
for i, r in enumerate(results):                     # runs with both fractions changed
    if i not in plotted:
        ax1.scatter([r[0]], [means[i]], color="tab:green", marker="^", zorder=3,
                    label="R2 {:.2f} & R3 {:.2f}".format(r[0], r[1]))
        ax1.annotate("{:+.1f}%".format(change[i]), (r[0], means[i]), textcoords="offset points",
                     xytext=(0, -14), ha="center", fontsize=8, color="tab:green")
        ax2.plot(r[2], r[3], ":", color="tab:green", label="R2 {:.2f} & R3 {:.2f}".format(r[0], r[1]))

ax1.axhline(ref, color="0.7", lw=0.8, zorder=0)
ax1.set_xlabel("excreted fraction of TEP uptake")
ax1.set_ylabel("annual mean {} [{}]".format(var, units))
ax1.set_title("(a) {}, mean over {}".format(long_name, year), fontsize=10)
ax1.invert_xaxis()                                  # from 0.90 (case) to 0.10
ax1.legend(fontsize=8, frameon=False)
ax1.margins(y=0.12)

ax2.set_ylabel("{} [{}]".format(var, units))
ax2.set_title("(b) daily values in {}".format(year), fontsize=10)
ax2.legend(fontsize=7, frameon=False, ncol=2)
fig.autofmt_xdate()
plt.setp(ax1.get_xticklabels(), rotation=0, ha="center")
fig.tight_layout()

# Save to Cloud Storage: /tmp/data is cleared when the workflow finishes
results_folder = str(param_cloud_folder)
os.makedirs(results_folder, exist_ok=True)
figure_file = os.path.join(results_folder, "TEPexcretion_{}_{}.png".format(var, year))
fig.savefig(figure_file, dpi=150)
with open(os.path.join(results_folder, "TEPexcretion_{}_{}.csv".format(var, year)), "w") as f:
    f.write("\n".join(table) + "\n")
for run_dir in folders:                  # keep the model logs and run info as well
    name = os.path.basename(os.path.normpath(run_dir))
    for fname in ("log.out", "run_info.json"):
        if os.path.exists(os.path.join(run_dir, fname)):
            os.makedirs(os.path.join(results_folder, name), exist_ok=True)
            shutil.copy(os.path.join(run_dir, fname), os.path.join(results_folder, name, fname))
print("results_folder:", results_folder)
print("figure:", figure_file)
plt.show()